In [ ]:
import pandas as pd
import seaborn as sns

tips = sns.load_dataset("tips")
penguins = sns.load_dataset("penguins")  # has real missing values, good for Day 2

tips.head()

## Day 1 — Series, DataFrame, and Selection

**Concept:** A `Series` is one labeled column. A `DataFrame` is a table of them sharing an index. Every DataFrame has an index (row labels) and columns (column labels) — selection bugs almost always come from confusing the two. `.loc[]` selects by label, `.iloc[]` selects by position — mixing these up is the single most common pandas mistake.

In [ ]:
# Todo 1: explore the shape of the data
#
# df.head(n=5)     -> first n rows, no args needed, just call it: df.head()
# df.info()        -> column names, dtypes, non-null counts, all in one printout
# df.describe()    -> count/mean/std/min/max for numeric columns only
# df.shape         -> a (rows, columns) tuple, NOT a method — no parentheses
# df.dtypes        -> a Series mapping each column name to its dtype



In [ ]:
# Todo 2: select a single column (Series) vs a DataFrame with one column
#
# df["col_name"]     -> single square brackets, one column name (a string)
#                        returns a Series (1-dimensional)
# df[["col_name"]]   -> double square brackets, a LIST containing one name
#                        returns a DataFrame (2-dimensional, still has columns)
#
# Try both on "total_bill" and check the type with type(...)



In [ ]:
# Todo 3: select rows by position with .iloc, and by condition
#
# df.iloc[start:stop]        -> position-based, like list slicing, stop is EXCLUSIVE
#                                e.g. tips.iloc[0:5] gives you the first 5 rows
# df[df["col"] > value]      -> boolean filtering: df["col"] > value makes a
#                                True/False Series, and df[...] keeps only True rows



In [ ]:
# Todo 4: select specific rows AND columns at once with .loc
#
# df.loc[row_condition, column_list]
#   row_condition -> a boolean Series, e.g. df["sex"] == "Female"
#   column_list   -> a list of column name strings, e.g. ["total_bill", "tip"]
# .loc uses LABELS (column names, or index labels), .iloc uses POSITIONS (integers)



## Day 2 — Cleaning

**Concept:** Missing data shows up as `NaN`. `.isna()` finds it, `.dropna()` removes it, `.fillna()` fills it — drop vs fill is a judgment call, not a rule, and you should always know why you picked one. `.duplicated()` finds repeated rows. `.astype()` converts a column's type. The `.str` accessor (`df["col"].str.strip()`, `.str.lower()`) runs string methods across an entire column at once, instead of looping.

In [ ]:
# Todo 1: find how many missing values per column
#
# df.isna()          -> same-shaped DataFrame of True/False (True = missing)
# df.isna().sum()     -> .sum() on booleans treats True as 1, False as 0,
#                         so summing per column counts the missing values



In [ ]:
# Todo 2: try both approaches and compare resulting shapes
#
# a) df.dropna()                      -> drops any row with at least one NaN
#                                         (use subset=["col"] to only check certain columns)
# b) df["col"].fillna(value)          -> fills NaN with `value`
#    df["col"].median()               -> a single number, use it as the fill value
#    Combine: df["col"].fillna(df["col"].median())
#
# Check df.shape before/after (a) to see how many rows you lost



In [ ]:
# Todo 3: check for duplicate rows
#
# df.duplicated()        -> True/False Series, True where a row is a repeat
#                            of an earlier row (all columns match)
# df.duplicated().sum()  -> count of duplicate rows



In [ ]:
# Todo 4: convert species to a category dtype
#
# df["col"].astype("category")   -> returns a new Series with that dtype,
#                                    doesn't change df in place unless you assign it back:
#                                    df["col"] = df["col"].astype("category")
# Check df.dtypes again afterward to confirm it changed



## Day 3 — Transformation: groupby, merge, pivot

**Concept:** `groupby` is "split into groups, do something per group, combine the results" — e.g. average tip per day. `.apply()` runs a custom function per row/group when there's no built-in vectorized way (use it as a last resort — vectorized pandas ops are faster). `merge` combines two DataFrames on a shared key, like a SQL join — know `how="inner"` (only matching rows) vs `"left"`/`"right"` vs `"outer"` (everything, gaps filled with NaN). `pivot_table` reshapes long data into a summary table.

In [ ]:
# Todo 1: average total_bill per day
#
# df.groupby("col_to_group_by")["col_to_aggregate"].mean()
#   groupby("day")       -> groups rows that share the same "day" value
#   ["total_bill"]       -> picks which column to aggregate, after grouping
#   .mean()               -> the aggregation function (also try .sum(), .count())



In [ ]:
# Todo 2: multiple aggregations, grouped by two columns
#
# df.groupby(["col1", "col2"])["target_col"].agg(["mean", "count"])
#   groupby([...]) with a LIST      -> groups by the combination of both columns
#   .agg([...]) with a LIST         -> runs multiple aggregation functions at once,
#                                       result has one column per function



In [ ]:
# Todo 3: build two tiny DataFrames by hand, then merge them
#
# pd.merge(left_df, right_df, on="shared_column", how="left")
#   left_df, right_df  -> the two DataFrames, in order
#   on="col"           -> the shared key column both DataFrames use to match rows
#   how=                -> "inner" (only matches), "left" (keep all of left_df),
#                          "right" (keep all of right_df), "outer" (keep everything)
#
# Predict the row count BEFORE running it, then check if you were right

customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "name": ["Ali", "Sara", "Umer"]
})
orders = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "amount": [250, 400, 120]
})

# your merge here



In [ ]:
# Todo 4: pivot table
#
# df.pivot_table(values="col_to_summarize", index="row_grouping_col",
#                 columns="col_grouping_col", aggfunc="mean")
#   values     -> the column you're summarizing (numeric)
#   index      -> becomes the new row labels
#   columns    -> becomes the new column labels
#   aggfunc    -> how to combine values that land in the same cell



## Day 4 — Feature Engineering, Scaling, Encoding

**Concept:** Models using gradient descent or distance (neural nets, logistic regression, KNN) are sensitive to feature scale — a column ranging 0-100,000 can dominate one ranging 0-1 even if it's less predictive. `StandardScaler` fixes this by transforming each column to mean 0, std 1: for every value it computes `(x - column_mean) / column_std`. Tree-based models (decision trees, XGBoost) don't need this. Categorical columns need to become numeric before most models can use them: one-hot encoding (`pd.get_dummies`) makes a new binary column per category.

In [ ]:
# Todo 1: manually compute (x - mean) / std, then compare against StandardScaler
#
# manual:  (df["col"] - df["col"].mean()) / df["col"].std()
#          -> this is a vectorized operation, it applies to every row at once
#
# StandardScaler():
#   scaler = StandardScaler()                        -> create it first
#   scaler.fit_transform(df[["col"]])                 -> note the DOUBLE brackets,
#                                                          it expects a 2D input (a DataFrame,
#                                                          not a Series), even for one column
#   returns a numpy array, not a DataFrame — wrap it in pd.DataFrame(...) if you want it back as one
from sklearn.preprocessing import StandardScaler

# manual version here

# StandardScaler version here — confirm the numbers roughly match
# (small differences can come from std using a slightly different denominator by default)



In [ ]:
# Todo 2: one-hot encode day and sex
#
# pd.get_dummies(df, columns=["col1", "col2"], drop_first=True)
#   df                  -> the DataFrame (function on pd, not df.get_dummies())
#   columns=[...]       -> which categorical columns to encode
#   drop_first=True     -> drops one category per column to avoid redundant columns
#                          (if you know it's not "Female", and there's no "Other", it's "Male")
#
# Look at df.columns before and after to see what got added



In [ ]:
# Todo 3: put it together — split, then fit-scale-transform
#
# train_test_split(X, y, test_size=0.2, random_state=42)
#   X, y            -> features and target, as separate DataFrame/Series
#   test_size=0.2   -> fraction held out for testing
#   random_state=   -> fix this number so your split is reproducible
#   returns 4 things, in this order: X_train, X_test, y_train, y_test
#
# Then: scaler.fit_transform(X_train)   -> fit AND transform on train
#       scaler.transform(X_val)         -> transform ONLY (no fit) on val/test
# This is a dress rehearsal for the real data.py you'll write on Days 5-7.
from sklearn.model_selection import train_test_split

